In [0]:
dbutils.widgets.text("refund_table", "devolucion")
dbutils.widgets.text("market_table", "markets")


In [0]:
CATALOG = 'dbassociate'
BRONZE_SCHEMA = 'bronze'
SILVER_SCHEMA = 'silver'

In [0]:
refund_count_table = dbutils.jobs.taskValues.get(taskKey="ingestion_refund_retail", key="refund_count_table", debugValue=15)
refund_table = dbutils.widgets.get("refund_table")
market_table = dbutils.widgets.get("market_table")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
window = Window.partitionBy("devolucion_id").orderBy("ingestion_at")

df_refund_dedup = (
    spark.read
        .table(f'{CATALOG}.{BRONZE_SCHEMA}.{refund_table}')
        .withColumn("rank", F.row_number().over(window))
        .filter(F.col("rank") == 1)
        .drop("rank")
        
    )

In [0]:
dedup_count = df_refund_dedup.count()

In [0]:
assert dedup_count == refund_count_table

In [0]:
df_refund_approved = (
    df_refund_dedup
        .filter(F.col("estado_devolucion") == "Aprobada")
)

In [0]:
df_market = (
    spark.read
        .table(f'{CATALOG}.{BRONZE_SCHEMA}.{market_table}')
    )

In [0]:
df_refund_approved_with_market = (
    df_refund_approved
        .withColumn("id", (F.random(seed=42) * 5) + 1)
        .select(
            "devolucion_id",
            "order_id",
            "fecha_devolucion",
            "cliente_id",
            "motivo",
            "monto_devuelto",
            "estado_devolucion",
            F.col("id").cast("int").alias("id")
        )
        .withColumn("tienda_id", F.concat(F.lit("T"), F.lpad(F.col("id"), 2, "0")))
        .drop("id")
)

In [0]:
df_refund_approved_with_market.createOrReplaceTempView("refund_approved_market")
df_market.createOrReplaceTempView("market")

In [0]:
%sql 

CREATE TABLE IF NOT EXISTS dbassociate.silver.devolucion_aprobada (
    devolucion_id string,
    order_id string,
    fecha_devolucion date,
    cliente_id string,
    motivo string,
    monto_devuelto decimal(10,2),
    estado_devolucion string,
    tienda_id string,
    tienda_nombre string
)
USING DELTA
PARTITIONED BY (motivo)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW refund_approved_extend as
SELECT 
    r.*,
    m.nombre as tienda_nombre
FROM refund_approved_market as r
LEFT JOIN market as m
    ON r.tienda_id = m.tienda_id

In [0]:
%sql

SELECT * FROM refund_approved_extend

In [0]:
spark.sql(f"""
    MERGE INTO {CATALOG}.{SILVER_SCHEMA}.devolucion_aprobada tgt
    USING refund_approved_extend src
    ON tgt.devolucion_id = src.devolucion_id
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *         
""")